## בסיס שיתוף פעולה: branch ו-merge conflict אחד, מבוקר

**Branch** הוא קו התפתחות נפרד של הריפו — עותק-עבודה של ההיסטוריה שאפשר לנסות עליו דברים בלי לגעת ב-`main`. אם הניסוי הצליח, ממזגים (`merge`) אותו בחזרה; אם לא, פשוט זורקים את ה-branch, ו-`main` נשאר בדיוק כפי שהיה.

זה הופך שימושי במיוחד כששני אנשים (או שתי גרסאות של עצמכם) עורכים את **אותו קובץ, באותה שורה**, על שני branches שונים. כשמנסים למזג, Git לא יודע איזה שינוי "נכון" — הוא מסמן **merge conflict**, ומחכה שתחליטו בעצמכם.

קובץ עם קונפליקט מכיל סימונים מיוחדים:

```
<<<<<<< HEAD
התוכן ב-branch שאתם נמצאים עליו כרגע
=======
התוכן ב-branch שמנסים למזג פנימה
>>>>>>> partner
```

הפתרון: לערוך את הקובץ ידנית, לבחור (או לשלב) את התוכן הרצוי, למחוק את שלושת סימוני ה-`<<<`/`===`/`>>>`, ואז `git add` + `git commit` רגילים כדי לסיים את המיזוג.

### דוגמה: שני שותפי מעבדה, אותה שורה

שני שותפים עובדים על אותה פונקציית fit. שותף א׳ (על `main`) מוסיף הדפסת מקדמים. שותף ב׳ (על branch נפרד, `partner`, שהסתעף מ**אותה** נקודת התחלה) מוסיף חישוב שגיאות — **על אותה שורה בדיוק**.

In [ ]:
import tempfile, os

workdir = tempfile.mkdtemp(prefix="git_branch_")
os.chdir(workdir)

!git init -q -b main
!git config user.email "student@example.com"
!git config user.name "Student"
!git config color.ui false

In [ ]:
%%writefile fit.py
import numpy as np

def linear_fit(x, y):
    a, b = np.polyfit(x, y, deg=1)
    return a, b

In [ ]:
!git add fit.py
!git commit -q -m "פונקציית fit ראשונית"
!git branch partner

`git branch partner` יצר branch חדש בשם `partner`, מאותה נקודה בדיוק שבה אנחנו עכשיו. עכשיו, על `main`, שותף א׳ מוסיף הדפסה:

In [ ]:
%%writefile fit.py
import numpy as np

def linear_fit(x, y):
    a, b = np.polyfit(x, y, deg=1)
    print(f"a={a:.3f}, b={b:.3f}")
    return a, b

In [ ]:
!git add fit.py
!git commit -q -m "הוספת הדפסת מקדמי ההתאמה"

עכשיו עוברים ל-branch `partner`, **מהנקודה המקורית** (בלי ההדפסה של שותף א׳), ומוסיפים שם חישוב שגיאות — על אותה שורה שהייתה שם במקור:

In [ ]:
!git checkout -q partner

In [ ]:
%%writefile fit.py
import numpy as np

def linear_fit(x, y):
    a, b = np.polyfit(x, y, deg=1)
    cov = np.polyfit(x, y, deg=1, cov=True)[1]
    return a, b, cov

In [ ]:
!git add fit.py
!git commit -q -m "הוספת מטריצת השגיאות מההתאמה"

עכשיו חוזרים ל-`main` ומנסים למזג את `partner` פנימה:

In [ ]:
!git checkout -q main
!git merge partner -m "merge partner into main"

`CONFLICT` — בדיוק כמו שציפינו: שני ה-branches שינו את אותה שורה בדרכים שונות, ו-Git לא יכול להחליט לבד. נסתכל על הקובץ:

In [ ]:
!git status --short
!cat fit.py

`<<<<<<< HEAD` עד `=======` הוא מה שהיה ב-`main` (הדפסת המקדמים); מ-`=======` עד `>>>>>>> partner` הוא מה שהיה ב-`partner` (חישוב השגיאות). כאן הפתרון הטבעי הוא **לשלב את שניהם**: גם ההדפסה, גם השגיאות. נערוך את הקובץ (ידנית, כמו שהייתם עושים בעורך קוד), ונסיים את המיזוג:

In [ ]:
%%writefile fit.py
import numpy as np

def linear_fit(x, y):
    a, b = np.polyfit(x, y, deg=1)
    cov = np.polyfit(x, y, deg=1, cov=True)[1]
    print(f"a={a:.3f}, b={b:.3f}")
    return a, b, cov

In [ ]:
!git add fit.py
!git commit -q --no-edit
!git log --oneline --graph --all

`git commit --no-edit` (אחרי שפתרנו את הקונפליקט וסימנו את זה עם `git add`) מסיים את המיזוג עם הודעת ברירת המחדל. גרף ההיסטוריה (`--graph --all`) מראה בבירור איך שני קווי ההתפתחות (`main` ו-`partner`) התאחדו לקומיט מיזוג משותף אחד.

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "בסימוני קונפליקט, מה נמצא בין <<<<<<< HEAD לבין =======?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "התוכן מה-branch שממנו מגיעים לתוך המיזוג (partner, במקרה שלנו)", "correct": False, "feedback": "לא — זה בדיוק ההפך. זה מה שנמצא אחרי =======."},
            {"answer": "התוכן מה-branch שאתם נמצאים עליו כרגע (main, במקרה שלנו)", "correct": True, "feedback": "נכון — HEAD מציין את הנקודה שבה אתם נמצאים עכשיו."},
            {"answer": "שורות שנמחקו לגמרי ולא יופיעו בגרסה הסופית", "correct": False, "feedback": "לא — שתי הגרסאות עדיין קיימות, וצריך לבחור/לשלב ביניהן ידנית."},
            {"answer": "הערות שGit הוסיף אוטומטית ואינן חלק מהקוד עצמו", "correct": False, "feedback": "רק סימוני <<<</===/>>>> עצמם הם תוספת של Git; התוכן ביניהם הוא קוד אמיתי משני ה-branches, שצריך לערוך."}
        ]
    },
    {
        "question": "למה בדוגמה למעלה בכלל קרה קונפליקט, ולא מיזוג אוטומטי חלק?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי שני ה-branches שינו קבצים שונים לגמרי", "correct": False, "feedback": "לא — בדיוק ההפך: אם היו משנים קבצים שונים, Git היה ממזג אוטומטית בלי בעיה."},
            {"answer": "כי שני ה-branches שינו את אותן שורות באותו קובץ, בדרכים שונות זו מזו", "correct": True, "feedback": "נכון — זה בדיוק מה שגורם ל-Git לא לדעת איזו גרסה לבחור."},
            {"answer": "כי לא הרצתם git add לפני git merge", "correct": False, "feedback": "לא קשור — git add דרוש רק אחרי פתרון הקונפליקט, לא לפני הניסיון למזג."},
            {"answer": "קונפליקטים קורים תמיד, בכל מיזוג, ללא קשר לתוכן השינויים", "correct": False, "feedback": "לא נכון — רוב המיזוגים חלקים לגמרי; קונפליקט קורה רק כששתי הגרסאות מתנגשות באותן שורות."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

בנו תרחיש קונפליקט דומה בעצמכם: קובץ עם `ax.set_title("Fit Results")`. על `main`, שנו לכותרת `"Linear Fit -- Run 1"`. על branch נפרד בשם `alt`, שהסתעף **מאותה נקודה**, שנו את אותה שורה לכותרת `"Fit Results (Weighted)"`. מזגו את `alt` לתוך `main`, פתרו את הקונפליקט (בחרו כותרת אחת, או שלבו), וסיימו את המיזוג.

In [ ]:
# workdir2 = tempfile.mkdtemp(prefix="git_branch_practice_")
# os.chdir(workdir2)
# !git init -q -b main
# !git config user.email "student@example.com"
# !git config user.name "Student"
# !git config color.ui false
#
# כתבו כאן את כל הרצף: קומיט בסיס, branch, שינוי על main, שינוי על alt, merge, פתרון

`````{admonition} פתרון
:class: dropdown, tip
```python
workdir2 = tempfile.mkdtemp(prefix="git_branch_practice_")
os.chdir(workdir2)
!git init -q -b main
!git config user.email "student@example.com"; !git config user.name "Student"; !git config color.ui false
```
```python
%%writefile plot_config.py
title = "Fit Results"
```
```python
!git add plot_config.py
!git commit -q -m "כותרת ראשונית"
!git branch alt
```
```python
%%writefile plot_config.py
title = "Linear Fit -- Run 1"
```
```python
!git add plot_config.py
!git commit -q -m "main: עדכון כותרת"
```
```python
!git checkout -q alt
```
```python
%%writefile plot_config.py
title = "Fit Results (Weighted)"
```
```python
!git add plot_config.py
!git commit -q -m "alt: עדכון כותרת"
```
```python
!git checkout -q main
!git merge alt -m "merge alt into main"
!cat plot_config.py   # יציג את סימוני הקונפליקט
```
```python
%%writefile plot_config.py
title = "Fit Results (Weighted, Run 1)"
```
```python
!git add plot_config.py
!git commit -q --no-edit
!git log --oneline --graph --all
```
`````